# 02 - Extract Cal-Adapt WRF holdings (s3://cadcat/wrf/ucla)

UCLA WRF dynamical downscaling: **hourly precipitation** (culvert/stormwater intensity),
**snow water equivalent** (snow/avalanche exposure, rain-on-snow), and **near-surface
wind**, subset to the Tahoe bbox from the 9 km d02 domain.

Layout (verified 2026-07-29): one consolidated hourly Zarr store per
`<model>/<scenario>/1hr/all/<domain>` holding all variables on a curvilinear
Lambert-conformal grid. **Scenarios: historical + ssp370 only** - no WRF ssp245 runs
exist; document this asymmetry wherever WRF-derived metrics appear next to LOCA2.

- Wind `U10`/`V10` and SWE `SNOW` are aggregated to daily on extract.
- `RAINC`+`RAINNC` are run-accumulated totals: hourly rate = time-diff, clipped at 0
  (bucket resets), kept hourly. Hourly volume is why `precip_hourly.models` is a
  3-model subset in config.

The LOCA2-Hybrid CA product also lives in this bucket but is disabled by default -
it is the same 1/16 deg grid as the primary LOCA2 pull, CA-only (see METHODS.md).

In [ ]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("02_extract_caladapt")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

In [ ]:
import s3fs

CAD = cfg["sources"]["caladapt"]
WRF = CAD["wrf"]
FS = s3fs.S3FileSystem(anon=True)

cad_dir = RAW / "caladapt"
cad_dir.mkdir(exist_ok=True)

# discover which model/scenario stores exist
available = {}
for model in WRF["models"]:
    try:
        scens = [k.rsplit("/", 1)[1] for k in FS.ls(f"{WRF['s3_prefix']}/{model}")]
    except FileNotFoundError:
        log.warning(f"[wrf] no S3 path for model {model}")
        continue
    available[model] = scens
log.info(f"[wrf] stores: { {m: s for m, s in available.items()} }")


def open_wrf(model, scenario):
    store = f"{WRF['s3_prefix']}/{model}/{scenario}/1hr/all/{WRF['domain']}"
    return xr.open_zarr(FS.get_mapper(store), consolidated=True), store


def wrf_bbox(ds):
    """Curvilinear subset: mask on 2-D lat/lon, then drop all-empty rows/cols."""
    mask = ((ds.lat >= BBOX["lat_min"]) & (ds.lat <= BBOX["lat_max"])
            & (ds.lon >= BBOX["lon_min"]) & (ds.lon <= BBOX["lon_max"]))
    return ds.where(mask, drop=True)

## Daily wind and SWE
Every configured model x available scenario: subset, aggregate hourly to daily
(mean + max for wind components, daily mean for SWE), save one NetCDF per group.

In [ ]:
for model, scens in available.items():
    for scen in scens:
        out_wind = cad_dir / f"wrf_{model}_{scen}_wind_daily__tahoe.nc"
        out_swe = cad_dir / f"wrf_{model}_{scen}_swe_daily__tahoe.nc"
        if (out_wind.exists() and out_swe.exists()
                and not cfg["run"]["overwrite_downloads"]):
            log.info(f"[wrf] {model}/{scen} wind+swe present - skipped")
            continue
        try:
            ds, store = open_wrf(model, scen)
            sub = wrf_bbox(ds[WRF["wind"]["variables"] + WRF["swe"]["variables"]])

            wind = xr.Dataset({
                "U10_mean": sub["U10"].resample(time="1D").mean(),
                "V10_mean": sub["V10"].resample(time="1D").mean(),
                "wspd_max": np.hypot(sub["U10"], sub["V10"]).resample(time="1D").max(),
            }).load()
            wind.to_netcdf(out_wind)
            append_manifest({"file": str(out_wind.relative_to(ROOT)),
                             "source_url": f"s3://{store}",
                             "size_bytes": out_wind.stat().st_size,
                             "sha256": sha256_file(out_wind),
                             "retrieved_date": str(pd.Timestamp.today().date()),
                             "notebook": "02_extract_caladapt"})

            swe = sub["SNOW"].resample(time="1D").mean().to_dataset(name="swe").load()
            swe.to_netcdf(out_swe)
            append_manifest({"file": str(out_swe.relative_to(ROOT)),
                             "source_url": f"s3://{store}",
                             "size_bytes": out_swe.stat().st_size,
                             "sha256": sha256_file(out_swe),
                             "retrieved_date": str(pd.Timestamp.today().date()),
                             "notebook": "02_extract_caladapt"})
            log.info(f"[wrf] {model}/{scen}: wind {out_wind.stat().st_size/1e6:.1f} MB, "
                     f"swe {out_swe.stat().st_size/1e6:.1f} MB")
        except Exception as e:
            log.warning(f"[wrf] FAILED {model}/{scen} wind/swe: {e}")

## Hourly precipitation
`precip_hourly.models` only (volume). Hourly rate = diff of (RAINC + RAINNC) along
time, negatives (accumulator resets) clipped to 0. First timestep is dropped by the
diff - one hour lost per run, irrelevant at climate scale.

In [ ]:
for model in WRF["precip_hourly"]["models"]:
    for scen in available.get(model, []):
        out = cad_dir / f"wrf_{model}_{scen}_prec_hourly__tahoe.nc"
        if out.exists() and not cfg["run"]["overwrite_downloads"]:
            log.info(f"[wrf-prec] {model}/{scen} present - skipped")
            continue
        try:
            ds, store = open_wrf(model, scen)
            sub = wrf_bbox(ds[WRF["precip_hourly"]["variables"]])
            total = (sub["RAINC"] + sub["RAINNC"])
            hourly = total.diff("time").clip(min=0).to_dataset(name="prec_mm_hr").load()
            hourly["prec_mm_hr"].attrs["units"] = "mm/hr"
            hourly.to_netcdf(out, encoding={"prec_mm_hr": {"zlib": True, "complevel": 4}})
            append_manifest({"file": str(out.relative_to(ROOT)),
                             "source_url": f"s3://{store}",
                             "size_bytes": out.stat().st_size,
                             "sha256": sha256_file(out),
                             "retrieved_date": str(pd.Timestamp.today().date()),
                             "notebook": "02_extract_caladapt"})
            log.info(f"[wrf-prec] {model}/{scen}: {out.stat().st_size/1e6:.1f} MB")
        except Exception as e:
            log.warning(f"[wrf-prec] FAILED {model}/{scen}: {e}")

log.info("Cal-Adapt WRF extract complete")